In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import jieba  #jieba分词
import os   
from sklearn.feature_extraction.text import TfidfVectorizer #TF-IDF
from sklearn.model_selection import GridSearchCV            #网格搜索
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix,classification_report #预测效果的评估
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif']=['SimHei']  #这两行用于plt图像显示中文，否则plt无法显示中文
plt.rcParams['axes.unicode_minus']=False

In [ ]:
!pip3 install opencc-python-reimplemented

In [ ]:
# 使用pandas读取数据集
import pandas as pd
# train = pd.read_table('train.txt', sep='\t',header=None)  # 训练集 占RAM太多了，當TFIDF矩陣的時候：(news_tfidf.toarray())
train = pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/train.txt', sep='\t', header=None)  # 训练集'dev.txt'，仍占用太多，限制只讀20000筆
# test = pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/test.txt', sep='\t',header=None, nrows=20000) # 测试集'test.txt'，仍占用太多，限制只讀20000筆

In [ ]:
# 添加列名便于对数据进行更好处理
train.columns = ["text_a",'label']
# dev.columns = ["text_a",'label']
# test.columns = ["text_a"]

In [ ]:
# 查看训练集
train

In [ ]:
# 查看训练集
train

##遍歷 train 變數的每一行，然後將每一行的簡體字轉換為繁體字

In [ ]:
from opencc import OpenCC

cc = OpenCC('s2tw')
for index, row in train.iterrows():
  train.loc[index, 'label'] = cc.convert(str(row['label']))
  train.loc[index, 'text_a'] = cc.convert(row['text_a'])
# for index, row in test.iterrows():
#   #test.loc[index, 'label'] = cc.convert(str(row['label']))
#   test.loc[index, 'text_a'] = cc.convert(row['text_a'])

In [ ]:
tmp=train.copy()#tmp是最原初的資料

In [ ]:
# 例如，如果您要將 csv 檔案中所有的 ' 字符替換為 _ 字符，您可以使用以下代碼：
# train = train.replace({'\'': '_'}, regex=True)
# 或者如果您要替換所有的錯誤字符, 您可以使用正規表達式來替換：
# import re
# train = train.apply(lambda x: re.sub(r'[^a-zA-Z0-9\n\.\?,!]+', ' ', x))
# 在上面這個例子中, 我們使用了正規表達式來過濾出非法字符, 只保留 a-z, A-Z, 0-9, .,?,!, 斷行符。

# 使用 replace() 函數後, 您可以將更新過後的資料寫入 csv 檔案中, 或者在 WEKA 中重新讀取資料。

# 記得使用前先將資料讀入, 並且使用 inplace=True 參數可以直接將更新過後的資料寫回原來的 dataframe 中
train=tmp#tmp是最原初的資料
# train = train[train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==0]
train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==0]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')]
# 查看训练集
train

In [ ]:
# 例如，如果您要將 csv 檔案中所有的 ' 字符替換為 _ 字符，您可以使用以下代碼：
# train = train.replace({'\'': '_'}, regex=True)
# 或者如果您要替換所有的錯誤字符, 您可以使用正規表達式來替換：
# import re
# train = train.apply(lambda x: re.sub(r'[^a-zA-Z0-9\n\.\?,!]+', ' ', x))
# 在上面這個例子中, 我們使用了正規表達式來過濾出非法字符, 只保留 a-z, A-Z, 0-9, .,?,!, 斷行符。

# 使用 replace() 函數後, 您可以將更新過後的資料寫入 csv 檔案中, 或者在 WEKA 中重新讀取資料。

# 記得使用前先將資料讀入, 並且使用 inplace=True 參數可以直接將更新過後的資料寫回原來的 dataframe 中
train=tmp#tmp是最原初的資料
train = train[train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==1]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==0]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')]
train.dropna()
# 查看训练集
train

In [ ]:
# 例如，如果您要將 csv 檔案中所有的 ' 字符替換為 _ 字符，您可以使用以下代碼：
# train = train.replace({'\'': '_'}, regex=True)
# 或者如果您要替換所有的錯誤字符, 您可以使用正規表達式來替換：
# import re
# train = train.apply(lambda x: re.sub(r'[^a-zA-Z0-9\n\.\?,!]+', ' ', x))
# 在上面這個例子中, 我們使用了正規表達式來過濾出非法字符, 只保留 a-z, A-Z, 0-9, .,?,!, 斷行符。

# 使用 replace() 函數後, 您可以將更新過後的資料寫入 csv 檔案中, 或者在 WEKA 中重新讀取資料。

# 記得使用前先將資料讀入, 並且使用 inplace=True 參數可以直接將更新過後的資料寫回原來的 dataframe 中
train=tmp#tmp是最原初的資料
# train = train[train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==1]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==0]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')]

train = train[~train.text_a.str.contains(r'[a-zA-Z]')]
# train = train[~train.text_a.str.contains(r'[a-zA-Z0-9\.\?,!]')]
# 這段程式碼會使用 pandas 的 str.contains() 函數來檢查 text_a 欄位中是否包含英文字符 (使用正規表達式 [a-zA-Z] 來匹配)， ~ 表示反選，最後結果為所有不包含英文字符的資料。
train.dropna(inplace=True)
# 查看训练集
train

In [ ]:
# 例如，如果您要將 csv 檔案中所有的 ' 字符替換為 _ 字符，您可以使用以下代碼：
# train = train.replace({'\'': '_'}, regex=True)
# 或者如果您要替換所有的錯誤字符, 您可以使用正規表達式來替換：
# import re
# train = train.apply(lambda x: re.sub(r'[^a-zA-Z0-9\n\.\?,!]+', ' ', x))
# 在上面這個例子中, 我們使用了正規表達式來過濾出非法字符, 只保留 a-z, A-Z, 0-9, .,?,!, 斷行符。

# 使用 replace() 函數後, 您可以將更新過後的資料寫入 csv 檔案中, 或者在 WEKA 中重新讀取資料。

# 記得使用前先將資料讀入, 並且使用 inplace=True 參數可以直接將更新過後的資料寫回原來的 dataframe 中
train=tmp#tmp是最原初的資料
# train = train[train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==1]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==0]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')]
# train = train[~train.text_a.str.contains(r'[a-zA-Z]')]
train = train[~train.text_a.str.contains(r'[a-zA-Z0-9\.\?,!]')]

train.dropna()
# 查看训练集
train

In [ ]:
# 例如，如果您要將 csv 檔案中所有的 ' 字符替換為 _ 字符，您可以使用以下代碼：
# train = train.replace({'\'': '_'}, regex=True)
# 或者如果您要替換所有的錯誤字符, 您可以使用正規表達式來替換：
# import re
# train = train.apply(lambda x: re.sub(r'[^a-zA-Z0-9\n\.\?,!]+', ' ', x))
# 在上面這個例子中, 我們使用了正規表達式來過濾出非法字符, 只保留 a-z, A-Z, 0-9, .,?,!, 斷行符。

# 使用 replace() 函數後, 您可以將更新過後的資料寫入 csv 檔案中, 或者在 WEKA 中重新讀取資料。

# 記得使用前先將資料讀入, 並且使用 inplace=True 參數可以直接將更新過後的資料寫回原來的 dataframe 中
train=tmp#tmp是最原初的資料
# train = train[train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==1]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')==0]
# train = train[~train.text_a.str.contains(r'[^a-zA-Z0-9\n\.\?,!]+')]
# train = train[~train.text_a.str.contains(r'[a-zA-Z]')]
train = train[~train.text_a.str.contains(r'[a-zA-Z0-9\.\?,!]')]

train.dropna(inplace=True)
# 查看训练集
train

In [ ]:
tmp

In [ ]:
train.to_csv('train_繁中逗號區隔version.csv', sep=',', index=False,encoding="utf-8")#用於weka上